Codagh Project
==============

### Import

In [1]:
import os
import numpy as np
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import chess
from chess import pgn
from tqdm import tqdm

## Data preprocessing

### Loading data

In [2]:
def get_number_of_games(file_path):
    number_of_games = 0
    with open(file_path, 'r') as pgn_file:
        while True:
            if not pgn.skip_game(pgn_file):
                break
            number_of_games += 1
    return number_of_games
    

def load_pgn(file_path, offset):
    games = np.memmap(filename="../lib/data/npy/games.npy", dtype='object', mode="r+")
    with open(file_path, 'r') as pgn_file:
        i = offset
        while True:
            game = pgn.read_game(pgn_file)
            if game is None:
                break
            games[i] = game
            i += 1
    del games
    return i

files = [file for file in os.listdir("../lib/data/pgn") if file.endswith(".pgn")]
LIMIT_OF_FILES = min(len(files), 2)
number_of_games = 0
for file in tqdm(files[:LIMIT_OF_FILES]):
    number_of_games += get_number_of_games(f"../lib/data/pgn/{file}")

games = np.memmap(filename="../lib/data/npy/games.npy", dtype='object', mode="w+", shape=(number_of_games))
del games
offset = 0
for file in tqdm(files[:LIMIT_OF_FILES]):
    offset = load_pgn(f"../lib/data/pgn/{file}", offset)
games = np.memmap(filename="../lib/data/npy/games.npy", dtype='object', mode="r+")

100%|██████████| 2/2 [00:00<00:00,  4.85it/s]


In [3]:
print(f"Games parsed: {len(games)}")

Games parsed: 404


### Convert data into tensors

In [4]:
from ridoc import generate_nn_input, encode_moves

In [5]:
positions, moves = generate_nn_input(games)
print(f"Number of samples: {len(moves)}")

100%|██████████| 404/404 [00:00<00:00, 688.83it/s]

Number of samples: 37768


In [6]:
from jesinia import ChessDataset

dataset = ChessDataset(positions, moves)